# SFT-DataJudge Phase 1: Explore and Clean Data

这一部分只做数据资产整理：三个数据源分别读取、分别清洗、分别总结。后续 Teacher Judge 抽样也会从每个 source 单独抽，不把三个分布直接混在一起随机抽样。

In [1]:
from pathlib import Path
import hashlib
import json
import re

import pandas as pd
from datasets import load_from_disk

pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "SFT-DataJudge":
    PROJECT_ROOT = Path(r"\\ad.uillinois.edu\engr-ews\haoran27\微调\SFT-DataJudge")

LLAMA_DATA = PROJECT_ROOT.parent / "LLaMA-Factory" / "data"
MODEL_SCOPE_DATA = LLAMA_DATA / "qwen3-finetune-test"

COT_ZH_PATH = LLAMA_DATA / "CoT_Chinese_data.csv"
FINETOME_PATH = MODEL_SCOPE_DATA / "FineTome" / "train"
OPENMATH_PATH = MODEL_SCOPE_DATA / "OpenMathReasoning" / "cot"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CoT-ZH:", COT_ZH_PATH)
print("FineTome:", FINETOME_PATH)
print("OpenMathReasoning:", OPENMATH_PATH)

PROJECT_ROOT: u:\微调\SFT-DataJudge
CoT-ZH: u:\微调\LLaMA-Factory\data\CoT_Chinese_data.csv
FineTome: u:\微调\LLaMA-Factory\data\qwen3-finetune-test\FineTome\train
OpenMathReasoning: u:\微调\LLaMA-Factory\data\qwen3-finetune-test\OpenMathReasoning\cot


## Common Cleaning Helpers

这里的 clean 只是第一层规则清洗，不代表最终质量分。最终质量要交给 Teacher Judge 和小模型 scorer。

In [2]:
def normalize_text(value):
    if value is None or pd.isna(value):
        return ""
    text = str(value).replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def stable_hash(text, length=16):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:length]


def strip_think_block(text):
    raw = normalize_text(text)
    lowered = raw.lower()
    end_tag = "</think>"
    if end_tag in lowered:
        end_index = lowered.index(end_tag) + len(end_tag)
        return normalize_text(raw[end_index:]), True, len(raw)
    return raw, False, len(raw)


def looks_like_mojibake(text):
    if not text:
        return False
    markers = ("�", "Ã", "Â", "å", "æ", "ç", "è", "é", "ð")
    return sum(text.count(marker) for marker in markers) >= 3


def has_repeated_punctuation(text):
    return bool(re.search(r"([!?。！？,.，])\1{5,}", text))


def add_cleaning_columns(
    df,
    source,
    language,
    task_type,
    min_instruction_len=10,
    min_output_len=20,
    max_output_len=32000,
):
    out = df.copy().reset_index(drop=True)
    out["source_index"] = out.index
    out["source"] = source
    out["language"] = language
    out["task_type"] = task_type
    out["instruction"] = out["instruction"].map(normalize_text)
    out["output"] = out["output"].map(normalize_text)
    out["instruction_len"] = out["instruction"].str.len()
    out["output_len"] = out["output"].str.len()
    out["pair_hash"] = out.apply(
        lambda row: stable_hash(row["instruction"] + "\n<OUTPUT>\n" + row["output"]),
        axis=1,
    )
    out["id"] = out.apply(lambda row: f"{source}_{int(row['source_index']):08d}_{row['pair_hash']}", axis=1)
    pair_counts = out["pair_hash"].value_counts()

    def flags_for_row(row):
        flags = []
        if not row["instruction"]:
            flags.append("empty_instruction")
        if not row["output"]:
            flags.append("empty_output")
        if 0 < row["instruction_len"] < min_instruction_len:
            flags.append("short_instruction")
        if 0 < row["output_len"] < min_output_len:
            flags.append("short_output")
        if row["output_len"] > max_output_len:
            flags.append("long_output")
        if pair_counts[row["pair_hash"]] > 1:
            flags.append("duplicate_pair")
        if looks_like_mojibake(row["instruction"]) or looks_like_mojibake(row["output"]):
            flags.append("possible_mojibake")
        if has_repeated_punctuation(row["instruction"]) or has_repeated_punctuation(row["output"]):
            flags.append("repeated_punctuation")
        return sorted(set(flags))

    out["flags"] = out.apply(flags_for_row, axis=1)
    out["is_clean"] = out["flags"].map(len).eq(0)
    return out


def summarize_source(df, name):
    return {
        "source": name,
        "total": len(df),
        "clean": int(df["is_clean"].sum()),
        "clean_ratio": round(float(df["is_clean"].mean()), 4),
        "instruction_p50": int(df["instruction_len"].quantile(0.50)),
        "instruction_p95": int(df["instruction_len"].quantile(0.95)),
        "output_p50": int(df["output_len"].quantile(0.50)),
        "output_p95": int(df["output_len"].quantile(0.95)),
        "max_output_len": int(df["output_len"].max()),
    }


def flag_counts(df):
    return df.explode("flags")["flags"].dropna().value_counts().rename_axis("flag").reset_index(name="count")

## 1. CoT-ZH

中文推理问答数据，后续主要用来做中文 reasoning SFT 数据筛选。

In [3]:
df_cot_raw = pd.read_csv(COT_ZH_PATH)

print(df_cot_raw.shape)
print(df_cot_raw.columns.tolist())
display(df_cot_raw.isna().sum())
display(df_cot_raw.head(3))

(74771, 3)
['instruction', 'input', 'output']


instruction        0
input          74771
output             0
dtype: int64

,instruction,input,output
0,问题：卢比。 5600分为A、B、C三部分，如果A比C的比例是1/7:1/7:1/14，那么A比C多多少？\n选项：\n(A) 300\n(B) 992 \n(C) 1120\n(D) 552\n(E) 312 让我们先想想。一些随机推理：,NaN,1/7:1/7:1/14 = 2:2:1\n1/5*5600 = 1120\n2240-1120 = 1120 最终答案：(C)。
1,立方体的边长是 7a cm。找到它的表面？\n选项：\n(A) 24a8\n(B) 24a4\n(C) 24a1\n(D) 24a2\n(E) 294a2 嗯，我的意识流：,NaN,6a2 = 6 * 7a * 7a = 294a2 所以，答案是（E）。
2,一块 7 英尺的木板。 9英寸长分成3等份。每个部分的长度是多少？\n选项：\n(A) 31 英寸\n(B) 32 英寸\n(C) 33 英寸\n(D) 34 英寸\n(E) 35 英寸让我们先想想。意识流：,NaN,7 英尺 9 英寸是 84 + 9 = 93 英寸。所以 93/3 = 31 英寸或 2 英尺 7 英寸。\n所以，答案是（A）。


In [4]:
df_cot = df_cot_raw[["instruction", "output"]].copy()
df_cot = add_cleaning_columns(
    df_cot,
    source="cot_zh",
    language="zh",
    task_type="math_reasoning",
    min_instruction_len=10,
    min_output_len=20,
    max_output_len=32000,
)
df_cot_clean = df_cot[df_cot["is_clean"]].copy()

print("raw:", df_cot.shape, "clean:", df_cot_clean.shape)
display(df_cot[["instruction_len", "output_len"]].describe())
display(flag_counts(df_cot))
display(df_cot_clean.sample(5, random_state=42)[["instruction", "output", "instruction_len", "output_len"]])

raw: (74771, 12) clean: (72333, 12)


,instruction_len,output_len
count,74771.000000,74771.000000
mean,96.509262,51.325300
std,86.218497,41.693458
min,12.000000,4.000000
25%,70.000000,29.000000
50%,85.000000,36.000000
75%,103.000000,55.000000
max,14417.000000,1010.000000


,flag,count
0,short_output,1686
1,duplicate_pair,730
2,repeated_punctuation,18
3,possible_mojibake,13


,instruction,output,instruction_len,output_len
57813,给出一步一步的推理过程，然后给出最终答案。詹姆斯去宠物智能公司收养一只小狗。领养费是 200 美元，他的朋友同意支付其中的 25%。詹姆斯需要付出多少？,他的朋友同意支付 200 * .25 = 50 美元。所以他需要支付 200 - 50 = 150 美元。\n最终答案：150。,76,63
72949,问：如果您不是 T-Mobile 客户，可以使用 T-Mobile tuesdays 应用程序吗？现在，让我们一步一步地思考：,T-Mobile tuesdays 是一款面向 T-Mobile 用户的奖励应用程序。 T-Mobile Tuesdays 通过确保用户拥有 T-Mobile 电话号码来验证用户。\n答案是：没有。,63,98
27510,狮子座：我们可以从“一个大肚子的老人旁边是一辆装满东西的购物车和一条狗在街上”得出结论吗？那个“这个人把他所有的东西都放在购物车里，还有他的狗。”？\n选项：\n- 是\n- 否\n- 无法分辨\n美：好吧，那我们先想想……\n我：,购物车可能没有这个人所有的东西。狗可能不在购物车中。\n因此，答案是无法判断。,114,38
10223,您在休息时可以做什么会影响到其他人？\n选项：\n- 躺下\n- 入睡\n- 时间流逝\n- 打鼾\n- 大声\n请回答并提供答案解释。,大多数人在休息时打鼾。您可以在休息时打鼾，影响其他人。最终答案：打鼾。,67,35
47950,里奥：前提：“骑着他的越野车的人在玩把戏。”\n基于这个前提，我们能否得出结论：“这个人正在练习小轮车。”是真的吗？\n选项：\n- 是\n- 无法判断\n- 否\n美：好吧，那我们先想想……\n我：,该男子可能正在练习 BMX 以外的其他东西。\n因此，答案是无法判断。,98,34


## 2. FineTome-100k

英文高质量通用指令数据，自带 `score`，后续适合用作质量分布参考和 scorer 的弱监督信号。

In [5]:
ft_raw = load_from_disk(str(FINETOME_PATH))
print(ft_raw.column_names, len(ft_raw))
ft_raw[0]

['conversations', 'source', 'score'] 100000


{'conversations': [{'from': 'human',
   'value': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.'},
  {'from': 'gpt

In [6]:
def extract_finetome(example):
    convs = example.get("conversations") or []
    human = next((c.get("value", "") for c in convs if c.get("from") == "human"), "")
    gpt = next((c.get("value", "") for c in convs if c.get("from") == "gpt"), "")
    return {
        "instruction": human,
        "output": gpt,
        "original_source": example.get("source"),
        "original_score": example.get("score"),
    }


df_ft = pd.DataFrame([extract_finetome(x) for x in ft_raw])
df_ft = add_cleaning_columns(
    df_ft,
    source="finetome",
    language="en",
    task_type="general_instruction",
    min_instruction_len=10,
    min_output_len=20,
    max_output_len=32000,
)
df_ft_clean = df_ft[df_ft["is_clean"]].copy()

print("raw:", df_ft.shape, "clean:", df_ft_clean.shape)
display(df_ft[["instruction_len", "output_len", "original_score"]].describe())
display(flag_counts(df_ft))
display(df_ft_clean.sample(5, random_state=42)[["instruction", "output", "original_score", "instruction_len", "output_len"]])

raw: (100000, 14) clean: (96465, 14)


,instruction_len,output_len,original_score
count,100000.000000,100000.000000,100000.000000
mean,347.448580,1361.509340,3.965096
std,473.250498,839.260788,0.185928
min,3.000000,1.000000,3.735708
25%,92.000000,775.000000,3.825289
50%,167.000000,1185.000000,3.912496
75%,397.000000,1722.000000,4.066439
max,18210.000000,12474.000000,5.212621


,flag,count
0,duplicate_pair,3156
1,short_output,274
2,possible_mojibake,61
3,repeated_punctuation,48
4,short_instruction,4


,instruction,output,original_score,instruction_len,output_len
33514,How can I write a program that prints the Fibonacci series up to a specific number of terms using a while loop?,You can achieve this by implementing the following code:\n\ndef print_fibonacci_sequence():\n # Initialize the first two terms of the Fibonacci series\n fir...,4.005597,111,1549
797,"Write a function to check for palindrome\n\nGiven a string, the task is to determine whether it is a palindrome or not. A palindrome is a word, phrase, numb...",Here's the implementation of the function in Python:\n\n```python\ndef is_palindrome(string):\n # Convert the input string to lowercase\n string = string.lo...,4.562820,1370,1004
16186,How can I use Python to determine the day of the week for a given date?,"To find the day of the week for a specific date in Python, you can use the following code:\n\n```python\nimport datetime\nimport calendar\n\n# Get the date ...",4.147109,71,1249
29163,How can the unit circle be utilized to determine the values of the six trigonometric functions for an angle of 150 degrees?,"To find the values of the trigonometric functions for 150 degrees using the unit circle, first consider that the reference angle is 30 degrees. In the unit ...",4.034969,123,1091
38845,"Given a list of numbers, 1-10, write a program to determine the sum of the odd numbers.","Here's one possible solution in Python:\n\n```python\nnumbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]\nsum_of_odds = 0\n\nfor num in numbers:\n if num % 2 == 1:\n...",3.972187,87,681


## 3. OpenMathReasoning-mini

英文数学推理数据，有 `expected_answer` 和 `pass_rate_72b_tir`，后续适合做可验证 reasoning 数据和评估集来源。

In [7]:
omr_raw = load_from_disk(str(OPENMATH_PATH))
print(omr_raw.column_names, len(omr_raw))
omr_raw[0]

['expected_answer', 'problem_type', 'problem_source', 'generation_model', 'pass_rate_72b_tir', 'problem', 'generated_solution', 'inference_mode'] 19252


{'expected_answer': '14',
 'problem_type': 'has_answer_extracted',
 'problem_source': 'aops_c4_high_school_math',
 'generation_model': 'DeepSeek-R1',
 'pass_rate_72b_tir': '0.96875',
 'problem': 'Given $\\sqrt{x^2+165}-\\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.',
 'generated_solution': "<think>\nOkay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.\n\nFirst, let me write down the equation again to make sure I have it right:\n\n√(x² + 165) - √(x² - 52) = 7.\n\nOkay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:\n\n√(x² + 165) = 7 + √(x² - 52).\n\nNow, if I square both sides, maybe I can get rid of the square roots. Let's do that:\n\n(√(x² + 165))² = (7 + √(x² - 52))².\n\nSimplifying the left side:\n\nx² + 

In [8]:
df_omr = omr_raw.to_pandas()[
    ["problem", "generated_solution", "expected_answer", "problem_type", "pass_rate_72b_tir"]
].rename(columns={"problem": "instruction", "generated_solution": "output"})

omr_outputs = df_omr["output"].map(strip_think_block)
df_omr["output"] = omr_outputs.map(lambda x: x[0])
df_omr["raw_output_had_think"] = omr_outputs.map(lambda x: x[1])
df_omr["raw_output_len"] = omr_outputs.map(lambda x: x[2])

df_omr = add_cleaning_columns(
    df_omr,
    source="openmath_reasoning",
    language="en",
    task_type="math_reasoning",
    min_instruction_len=10,
    min_output_len=20,
    max_output_len=32000,
)
df_omr_clean = df_omr[df_omr["is_clean"]].copy()

print("raw:", df_omr.shape, "clean:", df_omr_clean.shape)
print("think blocks stripped:", int(df_omr["raw_output_had_think"].sum()))
display(df_omr[["instruction_len", "output_len", "raw_output_len", "pass_rate_72b_tir"]].describe())
display(flag_counts(df_omr))
display(df_omr_clean.sample(5, random_state=42)[["instruction", "output", "expected_answer", "pass_rate_72b_tir", "output_len"]])

raw: (19252, 17) clean: (19179, 17)
think blocks stripped: 19252


,instruction_len,output_len,raw_output_len
count,19252.00000,19252.000000,19252.000000
mean,145.81498,1810.327862,11647.129181
std,80.05984,591.297777,7805.753238
min,19.00000,457.000000,1627.000000
25%,89.00000,1394.000000,6119.750000
50%,128.00000,1746.000000,9219.000000
75%,179.00000,2145.000000,14936.000000
max,913.00000,6373.000000,68694.000000


,flag,count
0,repeated_punctuation,69
1,possible_mojibake,2
2,duplicate_pair,2


,instruction,output,expected_answer,pass_rate_72b_tir,output_len
18795,"Determine the values of $k$ for which the points $(k+7, 2)$, $(0, k-3)$, and $(3, 4)$ are vertices of a triangle.","To determine the values of \( k \) for which the points \((k+7, 2)\), \((0, k-3)\), and \((3, 4)\) are vertices of a triangle, we need to ensure that these ...",\( k \neq \frac{3 + \sqrt{145}}{2} \) and \( k \neq \frac{3 - \sqrt{145}}{2} \),0.96875,1524
7165,How many different rectangles can be drawn on an 8x8 chessboard?,"To determine the number of different rectangles that can be drawn on an 8x8 chessboard, we start by noting that a rectangle is defined by choosing two horiz...",1296,0.96875,875
15309,Factor the polynomial \(6x^3 - 13x^2 + 9x - 2\).,"To factor the polynomial \(6x^3 - 13x^2 + 9x - 2\), we start by identifying potential rational roots using the Rational Root Theorem. The possible rational ...",\((x-1)(2x-1)(3x-2)\),0.96875,1487
9672,Write \( A = 16^{0.249999999\ldots} \) as a fraction in simplest form.,"To solve the problem of expressing \( A = 16^{0.249999999\ldots} \) as a fraction in its simplest form, we follow these steps:\n\n1. **Identify the Exponent...",2/1,0.96875,1027
15960,"At the Rice Mathematics Tournament, 80% of contestants wear blue jeans, 70% wear tennis shoes, and 80% of those who wear blue jeans also wear tennis shoes. ...","To solve the problem, we need to determine the fraction of people wearing tennis shoes who are also wearing blue jeans. Let's denote the total number of con...",\(\frac{32}{35}\),0.96875,1237


## Phase 1 Summary

三个 source 分别 clean，最后只做 summary 对比。后续抽样仍然按 source 分开。

In [9]:
source_summary = pd.DataFrame([
    summarize_source(df_cot, "cot_zh"),
    summarize_source(df_ft, "finetome"),
    summarize_source(df_omr, "openmath_reasoning"),
])

display(source_summary)

total_summary = pd.DataFrame({
    "metric": ["total_records", "clean_records", "clean_ratio"],
    "value": [
        int(source_summary["total"].sum()),
        int(source_summary["clean"].sum()),
        round(source_summary["clean"].sum() / source_summary["total"].sum(), 4),
    ],
})
display(total_summary)

,source,total,clean,clean_ratio,instruction_p50,instruction_p95,output_p50,output_p95,max_output_len
0,cot_zh,74771,72333,0.9674,85,184,36,140,1010
1,finetome,100000,96465,0.9647,167,1270,1185,3043,12474
2,openmath_reasoning,19252,19179,0.9962,128,296,1746,2852,6373


,metric,value
0,total_records,194023.0000
1,clean_records,187977.0000
2,clean_ratio,0.9688


In [10]:
flag_summary = pd.concat(
    [
        flag_counts(df_cot).assign(source="cot_zh"),
        flag_counts(df_ft).assign(source="finetome"),
        flag_counts(df_omr).assign(source="openmath_reasoning"),
    ],
    ignore_index=True,
)[["source", "flag", "count"]]

display(flag_summary)

,source,flag,count
0,cot_zh,short_output,1686
1,cot_zh,duplicate_pair,730
2,cot_zh,repeated_punctuation,18
3,cot_zh,possible_mojibake,13
4,finetome,duplicate_pair,3156
5,finetome,short_output,274
6,finetome,possible_mojibake,61
7,finetome,repeated_punctuation,48
8,finetome,short_instruction,4
9,openmath_reasoning,repeated_punctuation,69


## Optional Export

这里可以把 notebook 里处理后的数据另存到 `data/processed/notebook_by_source/`。注意：后续 Teacher Judge 抽样应该从 per-source 文件开始。

In [11]:
EXPORT_OUTPUTS = True

if EXPORT_OUTPUTS:
    out_dir = PROJECT_ROOT / "data" / "processed" / "notebook_by_source"
    out_dir.mkdir(parents=True, exist_ok=True)

    keep_cols = [
        "id", "source", "source_index", "language", "task_type", "instruction", "output",
        "instruction_len", "output_len", "pair_hash", "flags", "is_clean",
    ]

    df_cot[keep_cols].to_json(out_dir / "cot_zh.jsonl", orient="records", lines=True, force_ascii=False)
    df_cot_clean[keep_cols].to_json(out_dir / "cot_zh_clean.jsonl", orient="records", lines=True, force_ascii=False)

    ft_cols = keep_cols + ["original_source", "original_score"]
    df_ft[ft_cols].to_json(out_dir / "finetome.jsonl", orient="records", lines=True, force_ascii=False)
    df_ft_clean[ft_cols].to_json(out_dir / "finetome_clean.jsonl", orient="records", lines=True, force_ascii=False)

    omr_cols = keep_cols + ["expected_answer", "problem_type", "pass_rate_72b_tir", "raw_output_had_think", "raw_output_len"]
    df_omr[omr_cols].to_json(out_dir / "openmath_reasoning.jsonl", orient="records", lines=True, force_ascii=False)
    df_omr_clean[omr_cols].to_json(out_dir / "openmath_reasoning_clean.jsonl", orient="records", lines=True, force_ascii=False)

    summary_dir = PROJECT_ROOT / "data" / "processed"
    source_summary.to_json(summary_dir / "notebook_source_summary.json", orient="records", force_ascii=False, indent=2)
    flag_summary.to_json(summary_dir / "notebook_flag_summary.json", orient="records", force_ascii=False, indent=2)

    print("saved to", out_dir)

saved to u:\微调\SFT-DataJudge\data\processed\notebook_by_source
